In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
B,T,C = 4,8,2               # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# 我們希望取得x[b,t]在T = 0~t的平均，作為預測下一個字符使用
xbow = torch.zeros((B,T,C))                 # bag of words

for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]

        # print(xprev)
        xbow[b,t] = torch.mean(xprev, 0)

print(xbow.shape)
print(xbow)

torch.Size([4, 8, 2])
tensor([[[ 3.7155e-01,  4.2899e-01],
         [-6.9379e-02,  1.1154e-02],
         [ 3.1820e-02, -2.8075e-01],
         [ 6.4578e-02,  1.5482e-01],
         [ 3.4421e-01,  1.8475e-01],
         [ 3.5015e-01, -2.1226e-01],
         [ 1.4435e-01, -1.3260e-01],
         [ 9.5503e-02, -2.1778e-01]],

        [[-6.9267e-03, -2.5255e-01],
         [-1.8575e-01, -7.5874e-01],
         [-5.1912e-02, -1.1311e+00],
         [ 3.1484e-01, -1.1844e+00],
         [-7.4345e-02, -1.1160e+00],
         [ 9.3679e-05, -9.5196e-01],
         [-2.4975e-01, -6.7622e-01],
         [-1.2186e-01, -5.2220e-01]],

        [[-1.9293e-01, -1.3944e+00],
         [ 1.0823e-01, -3.9621e-01],
         [ 2.3573e-01, -5.1666e-01],
         [ 2.8207e-01, -2.2773e-01],
         [ 1.7792e-01,  8.2975e-02],
         [ 2.8226e-01,  9.4980e-02],
         [ 1.2266e-01,  4.5945e-02],
         [-7.8171e-02, -1.0978e-02]],

        [[-8.9880e-04, -1.0029e+00],
         [-2.1459e-01,  1.2720e-01],
         [

In [8]:
# 利用矩陣運算較快求得平均
# 最原始狀況，利用全1方陣，計算各column的總和
a = torch.ones(3, 3)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=', a)
print('--')
print('b=', b)
print('--')
print('c=', c)

a= tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
--
b= tensor([[1., 7.],
        [2., 0.],
        [2., 6.]])
--
c= tensor([[ 5., 13.],
        [ 5., 13.],
        [ 5., 13.]])


In [ ]:
# 調整a成為下三角矩陣，求得序列前t個元素的總和
a = torch.tril(torch.ones(3, 3))
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=', a)
print('--')
print('b=', b)
print('--')
print('c=', c)

a= tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
--
b= tensor([[9., 9.],
        [1., 8.],
        [6., 3.]])
--
c= tensor([[ 9.,  9.],
        [10., 17.],
        [16., 20.]])


In [ ]:
# 將a矩陣針對各row取平均，相當於取得權重
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=', a)
print('--')
print('b=', b)
print('--')
print('c=', c)

a= tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b= tensor([[0., 6.],
        [8., 2.],
        [9., 1.]])
--
c= tensor([[0.0000, 6.0000],
        [4.0000, 4.0000],
        [5.6667, 3.0000]])


In [ ]:
# 使用矩陣乘法求T = 0~t的平均
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x


print(torch.allclose(xbow, xbow2))
print(xbow2)

True
tensor([[[ 3.7155e-01,  4.2899e-01],
         [-6.9379e-02,  1.1154e-02],
         [ 3.1820e-02, -2.8075e-01],
         [ 6.4578e-02,  1.5482e-01],
         [ 3.4421e-01,  1.8475e-01],
         [ 3.5015e-01, -2.1226e-01],
         [ 1.4435e-01, -1.3260e-01],
         [ 9.5503e-02, -2.1778e-01]],

        [[-6.9267e-03, -2.5255e-01],
         [-1.8575e-01, -7.5874e-01],
         [-5.1912e-02, -1.1311e+00],
         [ 3.1484e-01, -1.1844e+00],
         [-7.4345e-02, -1.1160e+00],
         [ 9.3680e-05, -9.5196e-01],
         [-2.4975e-01, -6.7622e-01],
         [-1.2186e-01, -5.2220e-01]],

        [[-1.9293e-01, -1.3944e+00],
         [ 1.0823e-01, -3.9621e-01],
         [ 2.3573e-01, -5.1666e-01],
         [ 2.8207e-01, -2.2773e-01],
         [ 1.7792e-01,  8.2975e-02],
         [ 2.8226e-01,  9.4980e-02],
         [ 1.2266e-01,  4.5945e-02],
         [-7.8171e-02, -1.0978e-02]],

        [[-8.9880e-04, -1.0029e+00],
         [-2.1459e-01,  1.2720e-01],
         [ 1.6253e-01, -1.4

In [ ]:
# 使用softmax處理
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
print(tril)

wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei)

wei = F.softmax(wei, dim=-1)
print(wei)

xbow3 = wei @ x
print(torch.allclose(xbow, xbow3))
print(xbow3)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000,

In [ ]:
# self-attention
torch.manual_seed(2048)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))

# single Head self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)
wei = q @ k.transpose(-2, -1)

# wei = q @ k.transpose(-2, -1) * head_size**-0.5

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei.shape)
print(wei)

# out = wei @ x
out = wei @ v

print(out.shape)
print(out)

torch.Size([4, 8, 8])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.8101, 0.1899, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2706, 0.6079, 0.1214, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3270, 0.1718, 0.1688, 0.3325, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3065, 0.0869, 0.0109, 0.5452, 0.0506, 0.0000, 0.0000, 0.0000],
         [0.0768, 0.0656, 0.4611, 0.1013, 0.2725, 0.0227, 0.0000, 0.0000],
         [0.2026, 0.2403, 0.1520, 0.0890, 0.0156, 0.2288, 0.0717, 0.0000],
         [0.0027, 0.0234, 0.0810, 0.0227, 0.8254, 0.0083, 0.0126, 0.0239]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5554, 0.4446, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1426, 0.6130, 0.2443, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0569, 0.1830, 0.3481, 0.4120, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6543, 0.0420, 0.0551, 0.2120, 0.0366, 0.0000, 0.0000, 0.0000],
 

In [ ]:
# layer normalization，基本上就是一般的正規化，只針對單個row處理
# BatchNorm則是針對一個批次的內容來處理
class LayerNorm1d: # (used to be BatchNorm1d)

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        # calculate the forward pass
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

In [23]:
torch.manual_seed(2468)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])